In [ ]:
# 📊 Data Handling
import pandas as pd
import numpy as np

# 📈 Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# ⚙️ Preprocessing & Scaling
from datetime import timedelta
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

# 📈 Analysis
import scipy.stats as stats
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
  
# 🧠 Modeling
from sklearn.cluster import KMeans, DBSCAN
import hdbscan
from sklearn.metrics import silhouette_score

# Part 0: Wallet list

Query:
https://github.com/bellatrix-ds/dune-sql-playground/blob/main/Big_Query/new_wallets_2024_onward.txt

 # Part 1: Basic Information

Query:
https://github.com/bellatrix-ds/dune-sql-playground/blob/main/Big_Query/filtered_EOA_tx_jan2024

In [ ]:
df1 = pd.read_csv('01_df_row.csv')

df1['first_tx'] = pd.to_datetime(df1['first_tx'])
df1['last_tx'] = pd.to_datetime(df1['last_tx'])

df1['eth_balance'] = pd.to_numeric(df1['eth_balance'], errors='coerce')
df1['eth'] =round(df1['eth_balance'] / 1e18)
df1 = df1.rename(columns={'address': 'wallet_address'})

df1 = df1.drop(['eth_balance'],axis=1)
df1.head()

# Part 2: Transaction Behavior

Query:
https://github.com/bellatrix-ds/dune-sql-playground/blob/main/Big_Query/wallet_activity_jan2024_onward

In [ ]:
df2 = pd.read_csv('02_df_row.csv')
df2['block_timestamp'] = pd.to_datetime(df2['block_timestamp'])
df2.head()

In [ ]:
wallet_stats = []

for address, group in df2.groupby('from_address'):
    group = group.sort_values('block_timestamp')
    group['value'] = pd.to_numeric(group['value'], errors='coerce')
    
    total_tx = len(group)
    avg_value = round(group['value'].mean()/ 1e18)
    
    if total_tx > 1:
        time_diffs = group['block_timestamp'].diff().dt.total_seconds() / 86400 
        avg_time_gap = avg_time_gap = time_diffs.mean() if total_tx > 1 else 0
    else:
        avg_time_gap = None

    min_date = group['block_timestamp'].min()
    max_date = group['block_timestamp'].max()
    total_days = (max_date - min_date).days + 1 
    weeks = total_days / 7 if total_days > 0 else 1
    months = total_days / 30 if total_days > 0 else 1

    tx_per_day = total_tx / total_days if total_days > 0 else total_tx
    tx_per_week = total_tx / weeks
    tx_per_month = total_tx / months

    wallet_stats.append({
        'wallet_address': address,
        'total_tx': total_tx,
        'avg_tx_value': avg_value,
        'tx_per_day': tx_per_day,
        'tx_per_week': tx_per_week,
        'tx_per_month': tx_per_month,
        'avg_time_gap_days': avg_time_gap
    })

result_df = pd.DataFrame(wallet_stats)

eth_price_usd = 3500

result_df['avg_value_usd'] = result_df['avg_tx_value'] * eth_price_usd

result_df['avg_time_gap_days'] = result_df['avg_time_gap_days'].apply(
    lambda x: 0 if pd.isna(x) or x < 1 else int(x)
)

result_df

# Final Data

In [ ]:
df_final = pd.merge(df1,result_df, on='wallet_address',how='inner')
df_final

# Part 3: Line Charts

## Query for number of tx and balance per month

https://github.com/bellatrix-ds/dune-sql-playground/blob/main/Big_Query/monthly_eth_balance_by_wallet

# Part 4:  Tables

## Query for the most intractions between EOA to EOA

https://github.com/bellatrix-ds/dune-sql-playground/blob/main/Big_Query/top_wallet_interactions_jan2024

## Query for the most used smart contracts EOA → Contract

https://github.com/bellatrix-ds/dune-sql-playground/blob/main/Big_Query/top_contract_interactions_jan2024